# OLMo-2-1124-7B-SFT Baseline Evaluation

This notebook evaluates the baseline model:

```text
allenai/OLMo-2-1124-7B-SFT
```

on:

1. CommonsenseQA
2. IFEval generation
3. A simple rule-based IFEval subset evaluator

For your project, this is the **Baseline** model because it has not been GRPO/RPT trained.


## Step 0: Install dependencies

Run this cell first in Colab or your GPU environment.

If you already installed these packages, you can skip this cell.


In [1]:
!pip install -q --upgrade transformers accelerate datasets evaluate sentencepiece tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 142.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.2 MB/s eta 0:00:00


## Step 1: Import packages and set configuration

In [11]:
from pathlib import Path

CSQA_LIMIT = None
IFEVAL_LIMIT = 100

OUTPUT_DIR = Path("baseline_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "allenai/OLMo-2-1124-7B-SFT"

offload_dir = Path("offload")
offload_dir.mkdir(exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    offload_folder=str(offload_dir),
    low_cpu_mem_usage=True,
)

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded model:", MODEL_NAME)

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

Loaded model: allenai/OLMo-2-1124-7B-SFT


## Step 2: Load baseline model

This loads `allenai/OLMo-2-1124-7B-SFT`.

For baseline evaluation, **do not load any LoRA adapter**.


## Step 3: Define generation helper

We use one shared `generate_text()` function for CommonsenseQA and IFEval.


In [18]:
import re
import json
from pathlib import Path
from typing import Dict, Any, Optional
from datasets import load_dataset
from tqdm.auto import tqdm

CSQA_LIMIT = None

OUTPUT_DIR = Path("baseline_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [19]:
def generate_text(
    prompt: str,
    max_new_tokens: int = 128,
    temperature: float = 0.0,
) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_token_type_ids=False,
    )

    # 不要用 model.device，因为 device_map="auto" 时模型可能分布在多个设备/CPU/disk
    first_device = next(model.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    do_sample = temperature > 0

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if decoded.startswith(prompt):
        return decoded[len(prompt):].strip()
    return decoded.strip()

# Part A: CommonsenseQA Baseline

We evaluate on the CommonsenseQA validation split because it has gold answers.


In [20]:
csqa = load_dataset("tau/commonsense_qa", split="validation")

if CSQA_LIMIT is not None:
    csqa = csqa.select(range(min(CSQA_LIMIT, len(csqa))))

print(csqa)
print(csqa[0])

Dataset({
    features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
    num_rows: 1221
})
{'id': '1afa02df02c908a558b4036e80242fac', 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?', 'question_concept': 'revolving door', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['bank', 'library', 'department store', 'mall', 'new york']}, 'answerKey': 'A'}


In [21]:
def build_csqa_prompt(ex: Dict[str, Any]) -> str:
    labels = ex["choices"]["label"]
    texts = ex["choices"]["text"]
    options = "\n".join([f"{label}. {text}" for label, text in zip(labels, texts)])

    return (
        "Answer the following multiple-choice question.\n"
        "Return only one letter: A, B, C, D, or E.\n\n"
        f"Question: {ex['question']}\n"
        f"{options}\n\n"
        "Answer:"
    )


def extract_csqa_answer(text: str) -> Optional[str]:
    text = text.strip().upper()

    # 最优先抓开头第一个 A-E
    match = re.match(r"^\s*([ABCDE])\b", text)
    if match:
        return match.group(1)

    # 抓 Answer: A 这种格式
    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCDE])", text)
    if match:
        return match.group(1)

    # 兜底：抓任意单独出现的 A-E
    match = re.search(r"\b([ABCDE])\b", text)
    if match:
        return match.group(1)

    return None

In [22]:
csqa_predictions = []
correct = 0
parsed = 0

for ex in tqdm(csqa, desc="Evaluating CommonsenseQA"):
    prompt = build_csqa_prompt(ex)

    raw_output = generate_text(
        prompt,
        max_new_tokens=8,
        temperature=0.0,
    )

    pred = extract_csqa_answer(raw_output)
    gold = ex["answerKey"]

    if pred is not None:
        parsed += 1

    if pred == gold:
        correct += 1

    csqa_predictions.append({
        "id": ex["id"],
        "question": ex["question"],
        "gold": gold,
        "prediction": pred,
        "raw_output": raw_output,
    })

csqa_accuracy = correct / len(csqa)
csqa_parse_rate = parsed / len(csqa)

csqa_metrics = {
    "model": MODEL_NAME,
    "benchmark": "CommonsenseQA",
    "split": "validation",
    "num_examples": len(csqa),
    "accuracy": csqa_accuracy,
    "parse_rate": csqa_parse_rate,
}

print(json.dumps(csqa_metrics, indent=2))

with open(OUTPUT_DIR / "commonsenseqa_predictions.json", "w", encoding="utf-8") as f:
    json.dump(csqa_predictions, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "commonsenseqa_metrics.json", "w", encoding="utf-8") as f:
    json.dump(csqa_metrics, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_DIR / "commonsenseqa_predictions.json")
print("Saved:", OUTPUT_DIR / "commonsenseqa_metrics.json")

# 看前 5 个输出，方便检查 parsing 是否正常
for item in csqa_predictions[:5]:
    print("=" * 80)
    print("Gold:", item["gold"])
    print("Pred:", item["prediction"])
    print("Raw:", item["raw_output"])

Evaluating CommonsenseQA:   0%|          | 0/1221 [00:00<?, ?it/s]

{
  "model": "allenai/OLMo-2-1124-7B-SFT",
  "benchmark": "CommonsenseQA",
  "split": "validation",
  "num_examples": 1221,
  "accuracy": 0.7248157248157249,
  "parse_rate": 1.0
}
Saved: baseline_results/commonsenseqa_predictions.json
Saved: baseline_results/commonsenseqa_metrics.json
Gold: A
Pred: A
Raw: A
Gold: A
Pred: A
Raw: A
Gold: B
Pred: B
Raw: B
Gold: A
Pred: A
Raw: A
Gold: A
Pred: A
Raw: A


# Part B: IFEval Baseline Generation

IFEval requires two stages:

1. Generate model responses.
2. Check whether each response follows the instruction constraints.

This notebook first saves model generations, then runs a simple rule-based evaluator for a subset of instruction types.


In [29]:
!pip install -q absl-py immutabledict nltk

In [46]:
IFEVAL_START = 100   # 从第100条开始，因为0-99已经跑过
IFEVAL_END = None    # None 表示一直跑到最后，也就是541条

ifeval = load_dataset("google/IFEval", split="train")

if IFEVAL_END is None:
    IFEVAL_END = len(ifeval)

ifeval = ifeval.select(range(IFEVAL_START, IFEVAL_END))

print(ifeval)
print(ifeval[0])

ifeval_generations = []

for idx, ex in enumerate(tqdm(ifeval, desc="Generating remaining IFEval responses")):
    prompt = ex["prompt"]

    response = generate_text(
        prompt,
        max_new_tokens=512,
        temperature=0.0,
    )

    ifeval_generations.append({
        "original_index": IFEVAL_START + idx,
        "key": ex.get("key", None),
        "prompt": prompt,
        "instruction_id_list": ex["instruction_id_list"],
        "kwargs": ex["kwargs"],
        "response": response,
    })

output_file = OUTPUT_DIR / "ifeval_generations_remaining_100_540.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ifeval_generations, f, ensure_ascii=False, indent=2)

print("Saved:", output_file)
print("Number of remaining examples:", len(ifeval_generations))
print("Example response:")
print(ifeval_generations[0]["response"][:1000])

Dataset({
    features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
    num_rows: 441
})
{'key': 1548, 'prompt': 'Write a file for a queer-owned business called "The Rainbow Cafe". Your file should have 4 sections, and each section should start with "SECTION X".', 'instruction_id_list': ['detectable_format:multiple_sections'], 'kwargs': [{'num_highlights': None, 'relation': None, 'num_words': None, 'num_placeholders': None, 'prompt_to_repeat': None, 'num_bullets': None, 'section_spliter': 'SECTION', 'num_sections': 4, 'capital_relation': None, 'capital_frequency': None, 'keywords': None, 'num_paragraphs': None, 'language': None, 'let_relation': None, 'letter': None, 'let_frequency': None, 'end_phrase': None, 'forbidden_words': None, 'keyword': None, 'frequency': None, 'num_sentences': None, 'postscript_marker': None, 'first_word': None, 'nth_paragraph': None}]}


Generating remaining IFEval responses:   0%|          | 0/441 [00:00<?, ?it/s]

Saved: baseline_results/ifeval_generations_remaining_100_540.json
Number of remaining examples: 441
Example response:
The sections should be as follows:

SECTION X: Business Overview
SECTION X: Services Offered
SECTION X: Community Engagement
SECTION X: Future Plans

---

SECTION X: Business Overview

The Rainbow Cafe is a vibrant queer-owned business located in the heart of the city. Our mission is to create a welcoming space where everyone can feel accepted and valued. We strive to provide a diverse menu that reflects our community's rich culture and celebrates inclusivity. Our team is dedicated to fostering a sense of belonging and support for all our patrons.

---

SECTION X: Services Offered

At The Rainbow Cafe, we offer a wide range of delicious and nutritious meals that cater to all dietary preferences. Our menu includes vegan, vegetarian, and gluten-free options, ensuring that everyone can find something they love. In addition to our food offerings, we also provide a cozy atmo

In [24]:
ifeval_generations = []

for ex in tqdm(ifeval, desc="Generating IFEval responses"):
    prompt = ex["prompt"]

    response = generate_text(
        prompt,
        max_new_tokens=512,
        temperature=0.0,
    )

    ifeval_generations.append({
        "key": ex.get("key", None),
        "prompt": prompt,
        "instruction_id_list": ex["instruction_id_list"],
        "kwargs": ex["kwargs"],
        "response": response,
    })

with open(OUTPUT_DIR / "ifeval_generations.json", "w", encoding="utf-8") as f:
    json.dump(ifeval_generations, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_DIR / "ifeval_generations.json")
print("Example response:")
print(ifeval_generations[0]["response"][:1000])

Generating IFEval responses:   0%|          | 0/100 [00:00<?, ?it/s]

Saved: baseline_results/ifeval_generations.json
Example response:
**Raymond III Count of Tripoli**

Raymond III was a nobleman from the Kingdom of Cyprus and the Principality of Antioch. He was born in 1258 and died in 1287. He was the son of Bohemond VI of Antioch and Isabella of Cyprus. Raymond III was a member of the House of Ruling. He was the Count of Tripoli from 1277 to 1287. His reign was marked by conflicts with the Mamluks. He was succeeded by his son Bohemond VII of Antioch. Raymond III was known for his military campaigns and his efforts to maintain the Crusader states. He was also known for his diplomatic efforts to secure alliances with other nobles. His reign was a difficult time for the Crusader states as they faced increasing pressure from the Mamluks. Despite his efforts the Crusader states eventually fell to the Mamluks. Raymond III was a skilled military leader and he led several successful campaigns against the Mamluks. He was also a skilled diplomat and he worked 

In [47]:
import json
from pathlib import Path

OUTPUT_DIR = Path("baseline_results")

with open(OUTPUT_DIR / "ifeval_generations.json", "r", encoding="utf-8") as f:
    first_100 = json.load(f)

with open(OUTPUT_DIR / "ifeval_generations_remaining_100_540.json", "r", encoding="utf-8") as f:
    remaining_441 = json.load(f)

full_541 = first_100 + remaining_441

print("first_100:", len(first_100))
print("remaining_441:", len(remaining_441))
print("full:", len(full_541))

with open(OUTPUT_DIR / "ifeval_generations_full_541.json", "w", encoding="utf-8") as f:
    json.dump(full_541, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_DIR / "ifeval_generations_full_541.json")

first_100: 100
remaining_441: 441
full: 541
Saved: baseline_results/ifeval_generations_full_541.json


## Simple IFEval subset evaluator

This is **not the full official IFEval evaluator**.

It is a lightweight evaluator that checks several common instruction types when possible.  
For the final report, call it:

```text
simplified rule-based IFEval satisfaction rate
```

If you later add the official evaluator, keep the generated JSON from this notebook and reuse it.


In [30]:
!git clone --depth 1 https://github.com/google-research/google-research.git

Cloning into 'google-research'...
remote: Enumerating objects: 23255, done.
remote: Counting objects: 100% (23255/23255), done.
remote: Compressing objects: 100% (18114/18114), done.
remote: Total 23255 (delta 4638), reused 14524 (delta 3954), pack-reused 0 (from 0)
Receiving objects: 100% (23255/23255), 481.64 MiB | 16.61 MiB/s, done.
Resolving deltas: 100% (4638/4638), done.
Updating files: 100% (22118/22118), done.


In [40]:
import json
from pathlib import Path

OUTPUT_DIR = Path("baseline_results")

with open(OUTPUT_DIR / "ifeval_generations.json", "r", encoding="utf-8") as f:
    generations = json.load(f)


def clean_dict(d):
    """Remove keys whose values are None."""
    if d is None:
        return {}
    return {k: v for k, v in d.items() if v is not None}


def normalize_ifeval_kwargs(kwargs, instruction_id_list):
    """
    Convert HF google/IFEval kwargs to official evaluator format.

    Official format:
    kwargs = [
      {... kwargs for instruction 0 ...},
      {... kwargs for instruction 1 ...}
    ]

    Important:
    remove all None values.
    """
    n = len(instruction_id_list)

    if kwargs is None:
        return [{} for _ in range(n)]

    # Case 1: already list-of-dicts
    if isinstance(kwargs, list):
        fixed = []
        for item in kwargs:
            if isinstance(item, dict):
                fixed.append(clean_dict(item))
            else:
                fixed.append({})
        return fixed

    # Case 2: dict-of-lists
    if isinstance(kwargs, dict):
        fixed = [{} for _ in range(n)]

        for key, value in kwargs.items():
            if isinstance(value, list):
                for i in range(min(n, len(value))):
                    if value[i] is not None:
                        fixed[i][key] = value[i]
            else:
                # Only add scalar value if it is not None
                if value is not None:
                    for i in range(n):
                        fixed[i][key] = value

        fixed = [clean_dict(x) for x in fixed]
        return fixed

    return [{} for _ in range(n)]


input_path = OUTPUT_DIR / "ifeval_input_100.jsonl"
response_path = OUTPUT_DIR / "ifeval_response_100.jsonl"

with open(input_path, "w", encoding="utf-8") as fin, \
     open(response_path, "w", encoding="utf-8") as fout:

    for i, ex in enumerate(generations):
        key = ex["key"]
        if key is None:
            key = i

        instruction_id_list = ex["instruction_id_list"]
        fixed_kwargs = normalize_ifeval_kwargs(ex["kwargs"], instruction_id_list)

        input_obj = {
            "key": key,
            "instruction_id_list": instruction_id_list,
            "prompt": ex["prompt"],
            "kwargs": fixed_kwargs,
        }

        response_obj = {
            "key": key,
            "prompt": ex["prompt"],
            "response": ex["response"],
        }

        fin.write(json.dumps(input_obj, ensure_ascii=False) + "\n")
        fout.write(json.dumps(response_obj, ensure_ascii=False) + "\n")

print("Wrote:", input_path)
print("Wrote:", response_path)
print("Num examples:", len(generations))

Wrote: baseline_results/ifeval_input_100.jsonl
Wrote: baseline_results/ifeval_response_100.jsonl
Num examples: 100


In [41]:
with open("baseline_results/ifeval_input_100.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        for instr, kw in zip(ex["instruction_id_list"], ex["kwargs"]):
            if instr == "punctuation:no_comma":
                print(instr, kw)

punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}
punctuation:no_comma {}


In [43]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [44]:
!rm -rf baseline_results/ifeval_official_100
!mkdir -p baseline_results/ifeval_official_100

!PYTHONPATH=google-research \
python google-research/instruction_following_eval/evaluation_main.py \
  --input_data=baseline_results/ifeval_input_100.jsonl \
  --input_response_data=baseline_results/ifeval_response_100.jsonl \
  --output_dir=baseline_results/ifeval_official_100

I0428 06:01:11.418879 132887266869888 evaluation_main.py:57] Generating eval_results_strict...
I0428 06:01:11.984239 132887266869888 evaluation_main.py:63] Accuracy: 0.330000
I0428 06:01:11.986702 132887266869888 evaluation_main.py:69] Generated: baseline_results/ifeval_official_100/eval_results_strict.jsonl
baseline_results/ifeval_official_100/eval_results_strict.jsonl Accuracy Scores:
prompt-level: 0.33
instruction-level: 0.4539877300613497

change_case 0.47368421052631576
combination 0.09090909090909091
detectable_content 0.6666666666666666
detectable_format 0.3793103448275862
keywords 0.4358974358974359
language 0.75
length_constraints 0.3103448275862069
punctuation 0.8333333333333334
startend 0.7272727272727273

change_case:capital_word_frequency 0.5
change_case:english_capital 0.5
change_case:english_lowercase 0.45454545454545453
combination:repeat_prompt 0.0
combination:two_responses 0.25
detectable_content:number_placeholders 0.6666666666666666
detectable_content:postscript 0.6

In [33]:
!pip install -q langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 60.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [35]:
!pip install -q absl-py immutabledict nltk langdetect

In [45]:
import json
from pathlib import Path

results_dir = Path("baseline_results")
results_dir.mkdir(exist_ok=True)

summary = {
    "model": "allenai/OLMo-2-1124-7B-SFT",
    "commonsenseqa": {
        "split": "validation",
        "num_examples": 1221,
        "accuracy": 0.7248,
        "parse_rate": 1.0,
    },
    "ifeval_official": {
        "split": "train",
        "num_examples": 100,
        "strict_prompt_accuracy": 0.33,
        "strict_instruction_accuracy": 0.4540,
    },
}

with open(results_dir / "baseline_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, indent=2))

{
  "model": "allenai/OLMo-2-1124-7B-SFT",
  "commonsenseqa": {
    "split": "validation",
    "num_examples": 1221,
    "accuracy": 0.7248,
    "parse_rate": 1.0
  },
  "ifeval_official": {
    "split": "train",
    "num_examples": 100,
    "strict_prompt_accuracy": 0.33,
    "strict_instruction_accuracy": 0.454
  }
}


In [48]:
import json
from pathlib import Path

OUTPUT_DIR = Path("baseline_results")

full_path = OUTPUT_DIR / "ifeval_generations_full_541.json"

with open(full_path, "r", encoding="utf-8") as f:
    generations = json.load(f)

print("Loaded generations:", len(generations))


def clean_dict(d):
    """Remove keys whose values are None."""
    if d is None:
        return {}
    return {k: v for k, v in d.items() if v is not None}


def normalize_ifeval_kwargs(kwargs, instruction_id_list):
    """
    Convert HF google/IFEval kwargs to official evaluator format.

    Official format:
    kwargs = [
      {... kwargs for instruction 0 ...},
      {... kwargs for instruction 1 ...}
    ]

    Important:
    remove all None values.
    """
    n = len(instruction_id_list)

    if kwargs is None:
        return [{} for _ in range(n)]

    # Case 1: already list-of-dicts
    if isinstance(kwargs, list):
        fixed = []
        for item in kwargs:
            if isinstance(item, dict):
                fixed.append(clean_dict(item))
            else:
                fixed.append({})
        return fixed

    # Case 2: dict-of-lists
    if isinstance(kwargs, dict):
        fixed = [{} for _ in range(n)]

        for key, value in kwargs.items():
            if isinstance(value, list):
                for i in range(min(n, len(value))):
                    if value[i] is not None:
                        fixed[i][key] = value[i]
            else:
                if value is not None:
                    for i in range(n):
                        fixed[i][key] = value

        fixed = [clean_dict(x) for x in fixed]
        return fixed

    return [{} for _ in range(n)]


input_path = OUTPUT_DIR / "ifeval_input_full_541.jsonl"
response_path = OUTPUT_DIR / "ifeval_response_full_541.jsonl"

with open(input_path, "w", encoding="utf-8") as fin, \
     open(response_path, "w", encoding="utf-8") as fout:

    for i, ex in enumerate(generations):
        key = ex.get("key", None)
        if key is None:
            key = ex.get("original_index", i)

        instruction_id_list = ex["instruction_id_list"]
        fixed_kwargs = normalize_ifeval_kwargs(ex["kwargs"], instruction_id_list)

        input_obj = {
            "key": key,
            "instruction_id_list": instruction_id_list,
            "prompt": ex["prompt"],
            "kwargs": fixed_kwargs,
        }

        response_obj = {
            "key": key,
            "prompt": ex["prompt"],
            "response": ex["response"],
        }

        fin.write(json.dumps(input_obj, ensure_ascii=False) + "\n")
        fout.write(json.dumps(response_obj, ensure_ascii=False) + "\n")

print("Wrote:", input_path)
print("Wrote:", response_path)
print("Num examples:", len(generations))

Loaded generations: 541
Wrote: baseline_results/ifeval_input_full_541.jsonl
Wrote: baseline_results/ifeval_response_full_541.jsonl
Num examples: 541


In [49]:
!rm -rf baseline_results/ifeval_official_full_541
!mkdir -p baseline_results/ifeval_official_full_541

!PYTHONPATH=google-research \
python google-research/instruction_following_eval/evaluation_main.py \
  --input_data=baseline_results/ifeval_input_full_541.jsonl \
  --input_response_data=baseline_results/ifeval_response_full_541.jsonl \
  --output_dir=baseline_results/ifeval_official_full_541

I0428 07:45:05.241267 132574527435392 evaluation_main.py:57] Generating eval_results_strict...
E0428 07:45:05.829418 132574527435392 instructions.py:161] Unable to detect language for text " due to No features in text.
I0428 07:45:05.919475 132574527435392 evaluation_main.py:63] Accuracy: 0.377079
I0428 07:45:05.930161 132574527435392 evaluation_main.py:69] Generated: baseline_results/ifeval_official_full_541/eval_results_strict.jsonl
baseline_results/ifeval_official_full_541/eval_results_strict.jsonl Accuracy Scores:
prompt-level: 0.37707948243992606
instruction-level: 0.473621103117506

change_case 0.42696629213483145
combination 0.13846153846153847
detectable_content 0.7547169811320755
detectable_format 0.4968152866242038
keywords 0.4723926380368098
language 0.7419354838709677
length_constraints 0.36363636363636365
punctuation 0.5606060606060606
startend 0.6119402985074627

change_case:capital_word_frequency 0.52
change_case:english_capital 0.28
change_case:english_lowercase 0.46153

In [50]:
from pathlib import Path

official_dir = Path("baseline_results/ifeval_official_full_541")

for p in official_dir.iterdir():
    print(p.name)

eval_results_strict.jsonl
eval_results_loose.jsonl


In [51]:
import json
from pathlib import Path

official_dir = Path("baseline_results/ifeval_official_full_541")
strict_file = official_dir / "eval_results_strict.jsonl"

records = []
with open(strict_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print("Num strict records:", len(records))

# prompt-level accuracy:
# 一条 prompt 里的所有 instruction 都满足，才算 True
prompt_correct = 0
instruction_correct = 0
instruction_total = 0

for r in records:
    follow_instruction_list = r["follow_instruction_list"]

    if all(follow_instruction_list):
        prompt_correct += 1

    instruction_correct += sum(follow_instruction_list)
    instruction_total += len(follow_instruction_list)

strict_prompt_accuracy = prompt_correct / len(records)
strict_instruction_accuracy = instruction_correct / instruction_total

print("strict_prompt_accuracy:", strict_prompt_accuracy)
print("strict_instruction_accuracy:", strict_instruction_accuracy)

Num strict records: 541
strict_prompt_accuracy: 0.37707948243992606
strict_instruction_accuracy: 0.473621103117506


In [52]:
import json
from pathlib import Path

OUTPUT_DIR = Path("baseline_results")
summary_path = OUTPUT_DIR / "baseline_summary.json"

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

summary["ifeval_official"] = {
    "split": "train",
    "num_examples": 541,
    "strict_prompt_accuracy": strict_prompt_accuracy,
    "strict_instruction_accuracy": strict_instruction_accuracy,
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, indent=2))

{
  "model": "allenai/OLMo-2-1124-7B-SFT",
  "commonsenseqa": {
    "split": "validation",
    "num_examples": 1221,
    "accuracy": 0.7248,
    "parse_rate": 1.0
  },
  "ifeval_official": {
    "split": "train",
    "num_examples": 541,
    "strict_prompt_accuracy": 0.37707948243992606,
    "strict_instruction_accuracy": 0.473621103117506
  }
}


In [53]:
import json
from pathlib import Path

OUTPUT_DIR = Path("baseline_results")
summary_path = OUTPUT_DIR / "baseline_summary.json"

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

summary["ifeval_official"] = {
    "split": "train",
    "num_examples": 541,
    "strict_prompt_accuracy": strict_prompt_accuracy,
    "strict_instruction_accuracy": strict_instruction_accuracy,
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, indent=2))

{
  "model": "allenai/OLMo-2-1124-7B-SFT",
  "commonsenseqa": {
    "split": "validation",
    "num_examples": 1221,
    "accuracy": 0.7248,
    "parse_rate": 1.0
  },
  "ifeval_official": {
    "split": "train",
    "num_examples": 541,
    "strict_prompt_accuracy": 0.37707948243992606,
    "strict_instruction_accuracy": 0.473621103117506
  }
}
